In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import kagglehub
import os


path = kagglehub.dataset_download("awaiskaggler/insurance-csv")
print("Path to dataset files:", path)
path_csv = os.path.join(path, "insurance.csv")


# load insurance.csv
df = pd.read_csv(path_csv)
print(df.head())





Using Colab cache for faster access to the 'insurance-csv' dataset.
Path to dataset files: /kaggle/input/insurance-csv
   age     sex   bmi  children smoker     region  expenses
0   19  female  27.9         0    yes  southwest  16884.92
1   18    male  33.8         1     no  southeast   1725.55
2   28    male  33.0         3     no  southeast   4449.46
3   33    male  22.7         0     no  northwest  21984.47
4   32    male  28.9         0     no  northwest   3866.86
age         0
sex         0
bmi         0
children    0
smoker      0
region      0
expenses    0
dtype: int64


In [ ]:
## Create label to classification

df['labels'] = (df['expenses'] > 15000 ).astype(int)


## seperate label and feature
x = df.drop(['labels'], axis = 1)
y = df['labels']

In [ ]:
## Data preprocessing

# clearly seperate data
num_cols = ['age','bmi','children']
cat_cols = ['sex','smoker','region']

# call methods of preprocessing
num_trans = StandardScaler()
cat_trans = OneHotEncoder(drop = 'first')

## start preprocessing

preprocessor = ColumnTransformer(
    transformers=[
        ('num',num_trans,num_cols),
        ('cat',cat_trans,cat_cols)
    ]
)


In [ ]:
## split data to train and test set

x_train, x_test, y_train, y_test = train_test_split(
    x,y ,test_size = 0.2, random_state=42, stratify=y
)

In [ ]:
## create pipeline

# SGDClassifier with logistic regression (binary classification)
clf = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('classifier', SGDClassifier(loss="log_loss", max_iter=1000, tol=1e-3, random_state=42))
])



## train model
clf.fit(x_train, y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'bmi', 'children']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first'),
                                                  ['sex', 'smoker',
                                                   'region'])])),
                ('classifier',
                 SGDClassifier(loss='log_loss', random_state=42))])

In [ ]:
# predictions
y_pred = clf.predict(x_test)

# accuracy
print("Accuracy:", accuracy_score(y_test, y_pred))

# confusion matrix
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# classification report
print("Classification Report:\n", classification_report(y_test, y_pred))


Accuracy: 0.914179104477612
Confusion Matrix:
 [[196   0]
 [ 23  49]]
Classification Report:
               precision    recall  f1-score   support

           0       0.89      1.00      0.94       196
           1       1.00      0.68      0.81        72

    accuracy                           0.91       268
   macro avg       0.95      0.84      0.88       268
weighted avg       0.92      0.91      0.91       268

